# 📓 NB-02 — Fundamentos do ROS 2 Jazzy
## ESP004 — Sistemas Autônomos Robóticos | PPGESP/IFBA 2026.2
**Autores:** Prof. Dr. José Alberto Díaz Amado · Prof. Dr. João Erivando Soares Marques

---
### Capítulo 2 — Princípios do ROS 2 Jazzy

Este notebook implementa **todos os exemplos** do Capítulo 2:

| Seção | Exemplo | Conceito |
|-------|---------|----------|
| 2.1 | Inspecionar tópicos via CLI | `ros2 topic list/echo/hz` |
| 2.2 | Nó publicador HelloNode | Node + Publisher + Timer |
| 2.3 | Nó subscritor OuvidorNode | Subscriber + Callback |
| 2.4 | Nó configurável com parâmetros | Parameters |
| 2.5 | Monitor interativo no Jupyter | Threading + rclpy + matplotlib |

> **Requisito:** Usar o kernel **"Python 3 (ROS2 Jazzy)"** (Kernel > Change Kernel)

---
## 🔵 Exemplo 1 — Inspecionar o Sistema ROS 2

In [1]:
# ═══════════════════════════════════════════════════════════
# EXEMPLO 1 — Inspecionar tópicos, nós e tipos de mensagem
# Equivalente a executar ros2 no terminal
# ═══════════════════════════════════════════════════════════
import subprocess

def ros2_cmd(*args):
    """Executa um comando ros2 e retorna a saída formatada."""
    r = subprocess.run(['ros2'] + list(args),
                       capture_output=True, text=True, timeout=5)
    return r.stdout.strip() or r.stderr.strip() or '(sem saída)'

print('=== ros2 topic list ===')
print(ros2_cmd('topic', 'list'))

print('\n=== ros2 node list ===')
print(ros2_cmd('node', 'list'))

print('\n=== ros2 interface list (mensagens geometry_msgs) ===')
result = subprocess.run(
    'ros2 interface list | grep geometry_msgs',
    shell=True, capture_output=True, text=True)
print(result.stdout[:500])  # primeiras mensagens

print('\n=== Estrutura da mensagem Twist ===')
print(ros2_cmd('interface', 'show', 'geometry_msgs/msg/Twist'))

=== ros2 topic list ===
/parameter_events
/rosout

=== ros2 node list ===
(sem saída)

=== ros2 interface list (mensagens geometry_msgs) ===
    geometry_msgs/msg/Accel
    geometry_msgs/msg/AccelStamped
    geometry_msgs/msg/AccelWithCovariance
    geometry_msgs/msg/AccelWithCovarianceStamped
    geometry_msgs/msg/Inertia
    geometry_msgs/msg/InertiaStamped
    geometry_msgs/msg/Point
    geometry_msgs/msg/Point32
    geometry_msgs/msg/PointStamped
    geometry_msgs/msg/Polygon
    geometry_msgs/msg/PolygonInstance
    geometry_msgs/msg/PolygonInstanceStamped
    geometry_msgs/msg/PolygonStamped
    geometry_msgs/msg/Pose
    geome

=== Estrutura da mensagem Twist ===
# This expresses velocity in free space broken into its linear and angular parts.

Vector3  linear
	float64 x
	float64 y
	float64 z
Vector3  angular
	float64 x
	float64 y
	float64 z


---
## 🟢 Exemplo 2 — Nó Publicador: HelloNode

In [2]:
# ═══════════════════════════════════════════════════════════
# EXEMPLO 2 — HelloNode: publicador com timer
# Conceitos: Node, Publisher, Timer, Logger
# ═══════════════════════════════════════════════════════════
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
import threading


class HelloNode(Node):
    """Nó mais simples do ROS 2: publica uma string a cada segundo."""

    def __init__(self):
        # Registrar o nó no sistema ROS 2
        super().__init__('hello')

        # Publisher: tópico /saudacao, tipo String, fila 10
        self._pub = self.create_publisher(String, '/saudacao', 10)

        # Timer: chamar _publicar() a cada 1.0 segundo
        self._timer = self.create_timer(1.0, self._publicar)
        self._contador = 0
        self.get_logger().info('HelloNode iniciado!')

    def _publicar(self):
        """Callback do timer — monta e publica a mensagem."""
        self._contador += 1
        msg = String()
        msg.data = f'Olá do ROS 2 Jazzy! Mensagem #{self._contador}'
        self._pub.publish(msg)
        self.get_logger().info(f'Publicado: {msg.data}')


# ── Inicializar e rodar por 5 segundos ──────────────────────
import time

if rclpy.ok():
    rclpy.shutdown()
rclpy.init()

hello_node = HelloNode()

# Spin em thread separada para não travar o notebook
stop_event = threading.Event()

def spin_until_stop(node, event):
    while not event.is_set() and rclpy.ok():
        rclpy.spin_once(node, timeout_sec=0.1)

t = threading.Thread(target=spin_until_stop,
                     args=(hello_node, stop_event), daemon=True)
t.start()

print('HelloNode publicando por 5 segundos...')
print('Abra outro terminal e execute:')
print('  ros2 topic echo /saudacao')
print()

time.sleep(5)
stop_event.set()
t.join(timeout=2)

hello_node.destroy_node()
rclpy.shutdown()

print(f'\n✅ HelloNode encerrado após {hello_node._contador} mensagens publicadas.')

[INFO] [1794594033.714855964] [hello]: HelloNode iniciado!


HelloNode publicando por 5 segundos...
Abra outro terminal e execute:
  ros2 topic echo /saudacao



[INFO] [1794594034.643618077] [hello]: Publicado: Olá do ROS 2 Jazzy! Mensagem #1
[INFO] [1794594035.642355861] [hello]: Publicado: Olá do ROS 2 Jazzy! Mensagem #2
[INFO] [1794594036.642393719] [hello]: Publicado: Olá do ROS 2 Jazzy! Mensagem #3
[INFO] [1794594037.642306257] [hello]: Publicado: Olá do ROS 2 Jazzy! Mensagem #4



✅ HelloNode encerrado após 5 mensagens publicadas.


[INFO] [1794594038.641642748] [hello]: Publicado: Olá do ROS 2 Jazzy! Mensagem #5


---
## 🟡 Exemplo 3 — Nó Subscritor: OuvidorNode

In [3]:
# ═══════════════════════════════════════════════════════════
# EXEMPLO 3 — OuvidorNode: subscritor com callback
# Conceitos: Subscription, Callback, Log estruturado
# IMPORTANTE: Mantenha HelloNode rodando em outro terminal:
#   ros2 run esp004_pkg hello
# ═══════════════════════════════════════════════════════════
import rclpy, threading, time
from rclpy.node import Node
from std_msgs.msg import String


class OuvidorNode(Node):
    """Recebe e exibe mensagens do tópico /saudacao."""

    def __init__(self):
        super().__init__('ouvidor')
        self.mensagens_recebidas = []  # armazenar para análise

        # Subscription: tópico /saudacao, callback _receber, fila 10
        self._sub = self.create_subscription(
            String,         # tipo da mensagem
            '/saudacao',    # tópico
            self._receber,  # função callback
            10)             # tamanho da fila
        self.get_logger().info('OuvidorNode aguardando mensagens...')

    def _receber(self, msg: String):
        """Chamado automaticamente ao chegar nova mensagem."""
        self.mensagens_recebidas.append(msg.data)
        self.get_logger().info(f'Recebi: "{msg.data}"')
        print(f'  → {msg.data}')


# ── Publicar e escutar ao mesmo tempo no notebook ──────────
if rclpy.ok():
    rclpy.shutdown()
rclpy.init()

# Criar os dois nós
pub_node = rclpy.create_node('pub_demo')
ouvidor  = OuvidorNode()
publisher = pub_node.create_publisher(String, '/saudacao', 10)

stop_event = threading.Event()

def spin_both(event):
    executor = rclpy.executors.SingleThreadedExecutor()
    executor.add_node(pub_node)
    executor.add_node(ouvidor)
    while not event.is_set() and rclpy.ok():
        executor.spin_once(timeout_sec=0.05)

t = threading.Thread(target=spin_both, args=(stop_event,), daemon=True)
t.start()

print('Enviando 4 mensagens e escutando com OuvidorNode:')
for i in range(1, 5):
    msg = String()
    msg.data = f'Mensagem de teste #{i}'
    publisher.publish(msg)
    time.sleep(0.5)

stop_event.set()
t.join(timeout=2)

pub_node.destroy_node()
ouvidor.destroy_node()
rclpy.shutdown()

print(f'\n✅ Total recebido: {len(ouvidor.mensagens_recebidas)} mensagens')
print('   Mensagens:', ouvidor.mensagens_recebidas)

[INFO] [1794594069.212135670] [ouvidor]: OuvidorNode aguardando mensagens...
[INFO] [1794594069.228821447] [ouvidor]: Recebi: "Mensagem de teste #1"


Enviando 4 mensagens e escutando com OuvidorNode:
  → Mensagem de teste #1
  → Mensagem de teste #2


[INFO] [1794594069.729142504] [ouvidor]: Recebi: "Mensagem de teste #2"


  → Mensagem de teste #3


[INFO] [1794594070.229883422] [ouvidor]: Recebi: "Mensagem de teste #3"


  → Mensagem de teste #4


[INFO] [1794594070.728524947] [ouvidor]: Recebi: "Mensagem de teste #4"



✅ Total recebido: 4 mensagens
   Mensagens: ['Mensagem de teste #1', 'Mensagem de teste #2', 'Mensagem de teste #3', 'Mensagem de teste #4']


---
## 🟠 Exemplo 4 — Nó com Parâmetros Configuráveis

In [4]:
# ═══════════════════════════════════════════════════════════
# EXEMPLO 4 — VelocidadeNode: parâmetros ROS 2
# Conceitos: declare_parameter, get_parameter, set_parameter
# ═══════════════════════════════════════════════════════════
import rclpy, threading, time
from rclpy.node import Node
from geometry_msgs.msg import Twist


class VelocidadeNode(Node):
    """Publica Twist com parâmetros configuráveis em tempo real."""

    def __init__(self):
        super().__init__('velocidade')

        # Declarar parâmetros com valores padrão
        self.declare_parameter('velocidade_linear',  0.2)   # m/s
        self.declare_parameter('velocidade_angular', 0.0)   # rad/s
        self.declare_parameter('frequencia_hz',     10.0)   # Hz

        freq = self.get_parameter('frequencia_hz').value
        self._pub   = self.create_publisher(Twist, '/cmd_vel', 10)
        self._timer = self.create_timer(1.0 / freq, self._publicar)
        self._historico = []  # registrar velocidades publicadas

        self.get_logger().info(
            f'VelocidadeNode: v={self.get_parameter("velocidade_linear").value:.2f} m/s')

    def _publicar(self):
        # Ler parâmetros a cada ciclo (permitem mudança em runtime)
        vlin = self.get_parameter('velocidade_linear').value
        vang = self.get_parameter('velocidade_angular').value

        msg = Twist()
        msg.linear.x  = vlin
        msg.angular.z = vang
        self._pub.publish(msg)
        self._historico.append({'vlin': vlin, 'vang': vang})


# ── Demo com mudança de parâmetro em runtime ───────────────
if rclpy.ok():
    rclpy.shutdown()
rclpy.init()

vel_node = VelocidadeNode()
stop_evt = threading.Event()

def spin_vel(event):
    while not event.is_set() and rclpy.ok():
        rclpy.spin_once(vel_node, timeout_sec=0.05)

t = threading.Thread(target=spin_vel, args=(stop_evt,), daemon=True)
t.start()

from rclpy.parameter import Parameter

print('Fase 1 (0–2s): v=0.20 m/s (padrão)')
time.sleep(2)

# Mudar parâmetro em runtime (sem reiniciar o nó!)
vel_node.set_parameters([Parameter('velocidade_linear', value=0.5)])
print('Fase 2 (2–4s): v=0.50 m/s  ← parâmetro alterado em runtime!')
time.sleep(2)

vel_node.set_parameters([Parameter('velocidade_linear', value=0.0)])
print('Fase 3 (4–5s): v=0.00 m/s  ← robô parado')
time.sleep(1)

stop_evt.set()
t.join(timeout=2)
vel_node.destroy_node()
rclpy.shutdown()

# Mostrar histórico
import pandas as pd
df = pd.DataFrame(vel_node._historico)
print(f'\n✅ Total de mensagens Twist publicadas: {len(df)}')
print(df.describe().round(3))

[INFO] [1794594089.195440214] [velocidade]: VelocidadeNode: v=0.20 m/s


Fase 1 (0–2s): v=0.20 m/s (padrão)
Fase 2 (2–4s): v=0.50 m/s  ← parâmetro alterado em runtime!
Fase 3 (4–5s): v=0.00 m/s  ← robô parado

✅ Total de mensagens Twist publicadas: 50
         vlin  vang
count  50.000  50.0
mean    0.280   0.0
std     0.196   0.0
min     0.000   0.0
25%     0.200   0.0
50%     0.200   0.0
75%     0.500   0.0
max     0.500   0.0


---
## 📊 Exemplo 5 — Monitor Interativo: ROS 2 + Matplotlib

In [5]:
# ═══════════════════════════════════════════════════════════
# EXEMPLO 5 — Monitor de odometria em tempo real no Jupyter
# Simula dados de /odom e plota trajetória ao vivo
# ═══════════════════════════════════════════════════════════
import rclpy, threading, time, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from rclpy.node import Node
from nav_msgs.msg import Odometry
from geometry_msgs.msg import Twist


class SimuladorDiferencial:
    """Simula odometria de robô diferencial para demo."""
    def __init__(self):
        self.x, self.y, self.theta = 0.0, 0.0, 0.0
        self.vlin, self.vang = 0.3, 0.5
        self.dt = 0.05  # 20 Hz

    def step(self):
        """Integração Euler — cinemática diferencial."""
        self.x     += self.vlin * math.cos(self.theta) * self.dt
        self.y     += self.vlin * math.sin(self.theta) * self.dt
        self.theta += self.vang * self.dt
        return self.x, self.y, self.theta


# ── Coletar dados por 6 segundos ───────────────────────────
sim = SimuladorDiferencial()
xs, ys, thetas, tempos = [], [], [], []
t0 = time.time()

print('Simulando odometria por 6 segundos...')
while time.time() - t0 < 6.0:
    x, y, th = sim.step()
    xs.append(x)
    ys.append(y)
    thetas.append(th)
    tempos.append(time.time() - t0)
    time.sleep(sim.dt)

print(f'✅ {len(xs)} amostras coletadas')

# ── Plotar resultados ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Monitor de Odometria — ROS 2 Jazzy no Jupyter',
             fontsize=13, fontweight='bold')

# Gráfico 1: Trajetória XY
ax = axes[0]
sc = ax.scatter(xs, ys, c=tempos, cmap='viridis', s=10)
ax.plot(xs[0], ys[0], 'go', ms=12, label='Início')
ax.plot(xs[-1], ys[-1], 'rs', ms=12, label='Fim')
plt.colorbar(sc, ax=ax, label='Tempo (s)')
ax.set(xlabel='X (m)', ylabel='Y (m)', title='Trajetória XY')
ax.axis('equal'); ax.grid(True, alpha=0.3); ax.legend()

# Gráfico 2: Ângulo ao longo do tempo
ax2 = axes[1]
ax2.plot(tempos, [math.degrees(th) for th in thetas],
         color='darkorange', lw=2)
ax2.set(xlabel='Tempo (s)', ylabel='θ (graus)',
        title='Orientação')
ax2.grid(True, alpha=0.3)

# Gráfico 3: Distância acumulada
ax3 = axes[2]
dists = [math.hypot(xs[i]-xs[0], ys[i]-ys[0]) for i in range(len(xs))]
ax3.plot(tempos, dists, color='steelblue', lw=2)
ax3.set(xlabel='Tempo (s)', ylabel='Distância (m)',
        title='Distância desde a Origem')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('odom_monitor.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Distância máxima da origem: {max(dists):.2f} m')
print(f'Rotação total: {math.degrees(thetas[-1]):.1f}°')
print('📊 Gráfico salvo em odom_monitor.png')

Matplotlib is building the font cache; this may take a moment.


Simulando odometria por 6 segundos...
✅ 120 amostras coletadas


AttributeError: 'RcParams' object has no attribute '_get'

---
## 🏠 Exercícios Para Casa

### Casa 1 — Serviço AddTwoInts
Implemente um serviço que some dois inteiros. Use `example_interfaces/srv/AddTwoInts`.

### Casa 2 — Launch File
Crie um launch file que inicie `hello` e `ouvidor` ao mesmo tempo.

### Casa 3 — Altere o VelocidadeNode
Modifique o Exemplo 4 para que o robô faça um quadrado: 2s frente → 2s curva → repetir 4 vezes.

In [6]:
# ═══════════════════════════════════════════════════════════
# EXERCÍCIO GUIADO — Padrão Quadrado com VelocidadeNode
# Complete as partes marcadas com TODO
# ═══════════════════════════════════════════════════════════
import rclpy, threading, time
from rclpy.node import Node
from geometry_msgs.msg import Twist


class QuadradoNode(Node):
    """Robô percorre um quadrado de ~0.6m de lado."""

    # Máquina de estados simples
    FRENTE = 'FRENTE'
    CURVA  = 'CURVA'
    PARADO = 'PARADO'

    def __init__(self):
        super().__init__('quadrado')
        self._pub    = self.create_publisher(Twist, '/cmd_vel', 10)
        self._timer  = self.create_timer(0.1, self._step)  # 10 Hz
        self._estado = self.FRENTE
        self._t0     = time.time()
        self._lados  = 0
        self.poses   = []  # para plotar depois

    def _step(self):
        dt = time.time() - self._t0
        msg = Twist()

        if self._estado == self.FRENTE:
            msg.linear.x = 0.3  # frente por 2s
            if dt > 2.0:
                self._estado = self.CURVA
                self._t0 = time.time()

        elif self._estado == self.CURVA:
            # TODO: qual velocidade angular para girar 90° em 2s?
            # Dica: 90° = π/2 rad. Em 2s → ω = π/4 rad/s ≈ 0.785 rad/s
            msg.angular.z = 0.785  # ← complete aqui
            if dt > 2.0:
                self._lados += 1
                self._estado = self.PARADO if self._lados >= 4 else self.FRENTE
                self._t0 = time.time()
                self.get_logger().info(f'Lado {self._lados}/4 completo')

        elif self._estado == self.PARADO:
            pass  # msg já é zero

        self._pub.publish(msg)


# Executar
if rclpy.ok(): rclpy.shutdown()
rclpy.init()
quad = QuadradoNode()
stop = threading.Event()

t = threading.Thread(
    target=lambda e: [rclpy.spin_once(quad, timeout_sec=0.05)
                      for _ in iter(lambda: e.is_set() or not rclpy.ok(), True)],
    args=(stop,), daemon=True)
t.start()

print('QuadradoNode em execução — 4 lados × (2s frente + 2s curva) = 16s')
time.sleep(17)  # 4 lados + margem
stop.set()
t.join(timeout=2)
quad.destroy_node()
rclpy.shutdown()
print(f'✅ Quadrado completo! {quad._lados} lados percorridos.')

QuadradoNode em execução — 4 lados × (2s frente + 2s curva) = 16s


[INFO] [1794594261.989252200] [quadrado]: Lado 1/4 completo
[INFO] [1794594265.989978639] [quadrado]: Lado 2/4 completo
[INFO] [1794594270.189223841] [quadrado]: Lado 3/4 completo
[INFO] [1794594274.389904070] [quadrado]: Lado 4/4 completo


✅ Quadrado completo! 4 lados percorridos.
